In [ ]:
import json
import random
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from pathlib import Path
from textwrap import fill

In [ ]:
# Load dataset
json_path = 'dataset_qwen_pe_top1000_captioned.json'
with open(json_path, 'r') as f:
    dataset = json.load(f)
print(f"Total samples in dataset: {len(dataset)}")

In [ ]:
def overlay_mask(image_path, mask_path, alpha=0.5):
    """Overlay mask on image with red color"""
    try:
        image = Image.open(image_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')
        
        if mask.size != image.size:
            mask = mask.resize(image.size, Image.LANCZOS)
        
        image_np = np.array(image)
        mask_np = np.array(mask)
        
        # Create red mask overlay
        mask_colored = np.zeros_like(image_np)
        mask_colored[:, :, 0] = mask_np
        
        mask_binary = (mask_np > 0).astype(float)
        mask_binary = np.expand_dims(mask_binary, axis=2)
        
        overlayed = image_np * (1 - alpha * mask_binary) + mask_colored * alpha * mask_binary
        overlayed = overlayed.astype(np.uint8)
        
        return Image.fromarray(overlayed)
    except Exception as e:
        print(f"Error processing image: {e}")
        return None

In [ ]:
def visualize_sample(sample, index=None):
    """Visualize: Input, Edited+Mask, Reference GT, Reference GT Crop with prompt"""
    print("=" * 80)
    if index is not None:
        print(f"Sample index: {index}")
    edit_type = sample.get('edit_type', 'Unknown')
    prompt = sample['prompt']
    print(f"Edit type: {edit_type}")
    print(f"\nPrompt: {prompt}")
    print("=" * 80)
    
    base_path = Path('pico-banana-400k-subject_driven/openimages')
    mask_path = base_path / sample['back_mask']
    ref_gt_path = base_path / sample['ref_gt']
    ref_gt_crop_path = base_path / sample['ref_gt_crop']
    
    edit_images = sample['edit_image']
    if not isinstance(edit_images, list):
        edit_images = [edit_images]
    before_path = base_path / edit_images[0] if edit_images else None
    after_path = base_path / sample['image']
    
    fig, axes = plt.subplots(1, 4, figsize=(28, 7))
    axes = axes.flatten()
    
    # Input
    if before_path and before_path.exists():
        axes[0].imshow(Image.open(str(before_path)).convert('RGB'))
        axes[0].set_title('Input', fontsize=14, fontweight='bold')
    else:
        axes[0].set_title('Input missing', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # Edited + mask
    after_title = 'Edited + Mask' if mask_path.exists() else 'Edited'
    if after_path.exists():
        if mask_path.exists():
            overlayed_after = overlay_mask(str(after_path), str(mask_path))
            if overlayed_after:
                axes[1].imshow(overlayed_after)
            else:
                axes[1].imshow(Image.open(str(after_path)).convert('RGB'))
        else:
            axes[1].imshow(Image.open(str(after_path)).convert('RGB'))
        axes[1].set_title(after_title, fontsize=14, fontweight='bold')
    else:
        axes[1].set_title('Edited missing', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    # Reference GT
    if ref_gt_path.exists():
        axes[2].imshow(Image.open(str(ref_gt_path)).convert('RGB'))
        axes[2].set_title('Reference GT', fontsize=14, fontweight='bold')
    else:
        axes[2].set_title('Reference GT missing', fontsize=14, fontweight='bold')
    axes[2].axis('off')
    
    # Reference GT Crop
    if ref_gt_crop_path.exists():
        axes[3].imshow(Image.open(str(ref_gt_crop_path)).convert('RGB'))
        axes[3].set_title('Reference GT Crop', fontsize=14, fontweight='bold')
    else:
        axes[3].set_title('Reference GT Crop missing', fontsize=14, fontweight='bold')
    axes[3].axis('off')
    
    # Prompt overlay at bottom center
    wrapped_prompt = fill(prompt, width=120)
    fig.text(0.5, -0.02, wrapped_prompt, ha='center', va='top', fontsize=11, 
             bbox=dict(facecolor='white', alpha=0.85, edgecolor='gray', linewidth=1, pad=10))
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.08)
    plt.show()

In [ ]:
# View one random sample
random_index = random.randint(0, len(dataset) - 1)
random_sample = dataset[random_index]
visualize_sample(random_sample, random_index)

In [6]:
# View samples grouped by edit_type
def view_samples_by_edit_type(samples_per_type=1):
    """Show a random subset of samples for each edit_type"""
    from collections import defaultdict
    
    grouped = defaultdict(list)
    for idx, sample in enumerate(dataset):
        grouped[sample.get('edit_type', 'Unknown')].append((idx, sample))
    
    for edit_type in sorted(grouped.keys()):
        samples = grouped[edit_type]
        print(f"\n{'#' * 80}")
        print(f"Edit Type: {edit_type} (total {len(samples)} samples, showing {min(samples_per_type, len(samples))})")
        print(f"{'#' * 80}\n")
        chosen = random.sample(samples, k=min(samples_per_type, len(samples)))
        for idx, sample in chosen:
            visualize_sample(sample, index=idx)
            
# Show one sample per edit_type
view_samples_by_edit_type(samples_per_type=1)

In [7]:
# View specific sample by index
def view_sample_by_index(index):
    """View sample at specific index"""
    if 0 <= index < len(dataset):
        visualize_sample(dataset[index], index)
    else:
        print(f"Error: Index {index} out of range [0, {len(dataset)-1}]")

# Example: view sample at index 0
view_sample_by_index(0)